In [ ]:
!pip install fair-esm

import torch
import esm
import pandas as pd
import numpy as np
import umap

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.1/93.1 kB 5.4 MB/s eta 0:00:00


In [ ]:
# Load ESM-2 model
model, alphabet = esm.pretrained.esm2_t33_650M_UR50D()
batch_converter = alphabet.get_batch_converter()
model.eval()  # disables dropout for deterministic results

# Prepare data (first 2 sequences from ESMStructuralSplitDataset superfamily / 4)
data = [
    ("protein1", "MKTVRQERLKSIVRILERSKEPVSGAQLAEELSVSRQVIVQDIAYLRSLGYNIVATPRGYVLAGG"),
    ("protein2", "KALTARQQEVFDLIRDHISQTGMPPTRAEIAQRLGFRSPNAAEEHLKALARKGVIEIVSGASRGIRLLQEE"),
    ("protein2 with mask","KALTARQQEVFDLIRD<mask>ISQTGMPPTRAEIAQRLGFRSPNAAEEHLKALARKGVIEIVSGASRGIRLLQEE"),
    ("protein3",  "K A <mask> I S Q"),
]
batch_labels, batch_strs, batch_tokens = batch_converter(data)
batch_lens = (batch_tokens != alphabet.padding_idx).sum(1)

# Extract per-residue representations (on CPU)
with torch.no_grad():
    results = model(batch_tokens, repr_layers=[33], return_contacts=True)
token_representations = results["representations"][33]

# Generate per-sequence representations via averaging
# NOTE: token 0 is always a beginning-of-sequence token, so the first residue is token 1.
sequence_representations = []
for i, tokens_len in enumerate(batch_lens):
    sequence_representations.append(token_representations[i, 1 : tokens_len - 1].mean(0))

Downloading: "https://dl.fbaipublicfiles.com/fair-esm/models/esm2_t33_650M_UR50D.pt" to /root/.cache/torch/hub/checkpoints/esm2_t33_650M_UR50D.pt
Downloading: "https://dl.fbaipublicfiles.com/fair-esm/regression/esm2_t33_650M_UR50D-contact-regression.pt" to /root/.cache/torch/hub/checkpoints/esm2_t33_650M_UR50D-contact-regression.pt


In [ ]:
df = pd.read_csv("rhea_steroid_smiles.txt", sep="\t")
df.head()

,Compound 1 [H][C@]12CC[C@]3([H])[C@]([H])(CC[C@]4(C)[C@@H](O)CC[C@@]34[H])[C@@]1(C)CC[C@@H](O)C2
0,Compound 2 [H][C@@]12CCC3=CC(=O)CC[C@]3(C)[C@...
1,Compound 3 [H][C@@]12CC[C@]([H])([C@H](C)CC[C...
2,Compound 4 *C1CCC2C3CCC4CCCCC4(C)C3[C@@H](O)C...
3,Compound 5 CC(C)CC[C@H](O)C(C)C1CCC2C3CCC4CCC...
4,Compound 6 [H][C@@]12C[C@H](O)CC[C@]1(C)[C@@]...


In [ ]:
df.columns

Index(['Entry', 'Reviewed', 'Entry Name', 'Organism', 'Protein names',
       'Length', 'Sequence', 'Annotation', 'Gene Names', 'Rhea ID',
       'ChEBI IDs', 'SMILES'],
      dtype='object')

In [ ]:
data = list(zip(df['Entry'], df['Sequence']))
# Remove any rows with missing sequences
data = [(name, seq) for name, seq in data if seq]


In [ ]:
# Load model (small ESM2 variant)
model, alphabet = esm.pretrained.esm2_t6_8M_UR50D()
batch_converter = alphabet.get_batch_converter()
model.eval().cpu()

Downloading: "https://dl.fbaipublicfiles.com/fair-esm/models/esm2_t6_8M_UR50D.pt" to /root/.cache/torch/hub/checkpoints/esm2_t6_8M_UR50D.pt
Downloading: "https://dl.fbaipublicfiles.com/fair-esm/regression/esm2_t6_8M_UR50D-contact-regression.pt" to /root/.cache/torch/hub/checkpoints/esm2_t6_8M_UR50D-contact-regression.pt


ESM2(
  (embed_tokens): Embedding(33, 320, padding_idx=1)
  (layers): ModuleList(
    (0-5): 6 x TransformerLayer(
      (self_attn): MultiheadAttention(
        (k_proj): Linear(in_features=320, out_features=320, bias=True)
        (v_proj): Linear(in_features=320, out_features=320, bias=True)
        (q_proj): Linear(in_features=320, out_features=320, bias=True)
        (out_proj): Linear(in_features=320, out_features=320, bias=True)
        (rot_emb): RotaryEmbedding()
      )
      (self_attn_layer_norm): LayerNorm((320,), eps=1e-05, elementwise_affine=True)
      (fc1): Linear(in_features=320, out_features=1280, bias=True)
      (fc2): Linear(in_features=1280, out_features=320, bias=True)
      (final_layer_norm): LayerNorm((320,), eps=1e-05, elementwise_affine=True)
    )
  )
  (contact_head): ContactPredictionHead(
    (regression): Linear(in_features=120, out_features=1, bias=True)
    (activation): Sigmoid()
  )
  (emb_layer_norm_after): LayerNorm((320,), eps=1e-05, elementwis

In [ ]:
# Assume df has a 'Sequence' and 'Entry' column
embeddings = []

batch_size = 4  # Reduce if needed

for i in range(0, len(data), batch_size):
    batch = data[i:i+batch_size]
    batch_labels, batch_strs, batch_tokens = batch_converter(batch)
    batch_lens = (batch_tokens != alphabet.padding_idx).sum(1)

    with torch.no_grad():
        results = model(batch_tokens, repr_layers=[6], return_contacts=False)
        token_reps = results["representations"][6]

    for j, seq_len in enumerate(batch_lens):
        emb = token_reps[j, 1:seq_len - 1].mean(0).numpy()
        embeddings.append(emb)

In [ ]:
df_valid = df.iloc[:len(embeddings)].copy()
df_valid['embedding'] = embeddings


In [ ]:
embeddings[0].shape

(320,)

In [ ]:
# Convert list of embeddings into a NumPy matrix
X = np.stack(df_valid['embedding'].values)

# Fit UMAP
reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, metric='cosine')
umap_coords = reducer.fit_transform(X)

# Add UMAP results back to dataframe
df_valid['UMAP_1'] = umap_coords[:, 0]
df_valid['UMAP_2'] = umap_coords[:, 1]

store embeddings in pickle file with entry name and embedding

In [ ]:
df_valid.head()

,Entry,Reviewed,Entry Name,Organism,Protein names,Length,Sequence,Annotation,Gene Names,Rhea ID,ChEBI IDs,SMILES,embedding,UMAP_1,UMAP_2
0,A0A1Y3VSR9,unreviewed,A0A1Y3VSR9_9BACT,Butyricimonas sp. An62,Choloylglycine hydrolase,362,MKQKLSITLWVILPLIAFFPRELKACTGITLKAKDGSCIVARTIEW...,1,B5G13_13515,NaN,NaN,NaN,"[0.06972206, -0.12230003, 0.15123817, 0.135149...",-2.753601,4.198223
1,A0A096DGB7,unreviewed,A0A096DGB7_FLAPL,Flavonifractor plautii 1_3_50AFAA,Choloylglycine hydrolase/NAAA C-terminal domai...,369,MDSNESRARALLGDLSAGCSAVAWETADGGHLWGRNFDFNRIAADS...,1,HMPREF9460_00913,NaN,NaN,NaN,"[-0.035128225, -0.10921955, -0.015683033, 0.05...",-6.234045,1.294657
2,A0A2S8GTJ2,unreviewed,A0A2S8GTJ2_9BACT,Blastopirellula marina,Choloylglycine hydrolase,334,MTFAGSLAVLACTRILWNDNDLAVVSGRTMDWPESTEPILTILPRG...,1,C5Y93_02690,NaN,NaN,NaN,"[0.047784768, -0.06448585, 0.20188767, 0.00552...",-3.689390,2.836949
3,A0A1Q6E5Z9,unreviewed,A0A1Q6E5Z9_9BACT,Alistipes sp. 56_11,Choloylglycine hydrolase,360,MKRKLVAVLVIAAAAAAWPQGAEACTGITLKAKDGAYVVARTIEWG...,1,BHV63_08645,NaN,NaN,NaN,"[0.12747556, -0.11847376, 0.22990689, 0.006286...",-0.781092,2.161024
4,M7X2M2,unreviewed,M7X2M2_9BACT,Mariniradius saccharolyticus AK6,Choloylglycine hydrolase,459,MKAYIVTLLLSLAFIECFANTAFFVHGTKGILAKNHDSKSGHGILI...,1,C943_02016,NaN,NaN,NaN,"[-0.03618196, -0.038847215, 0.24334729, 0.0277...",-5.641917,4.276597


In [ ]:
df_valid.to_csv('sequence_embeddings.csv')

In [ ]:
proteins = pd.read_csv('sequence_embeddings.csv')

In [ ]:
proteins[['Protein names']]

,Protein names
0,Sterol carrier protein 2 (EC 2.3.1.155) (EC 2....
1,Sterol carrier protein 2 (EC 2.3.1.155) (EC 2....
2,Sterol carrier protein 2 (EC 2.3.1.155) (EC 2....
3,Sterol carrier protein 2 (EC 2.3.1.155) (EC 2....
4,Sterol carrier protein 2 (EC 2.3.1.155) (EC 2....
...,...
5593,Protein 4.2 (P4.2) (Erythrocyte membrane prote...
5594,5-hydroxytryptamine receptor 2C (5-HT-2C) (5-H...
5595,Voltage-dependent R-type calcium channel subun...
5596,Guanine nucleotide-binding protein subunit alp...


In [ ]:
import re

# Example assuming your dataframe is named df and column is 'Protein names'

# Remove everything in parentheses (including the parentheses)
df['base_name'] = proteins['Protein names'].apply(lambda x: re.sub(r"\s*\(.*?\)", "", x))

# Strip extra spaces
df['base_name'] = df['base_name'].str.strip()

# Get unique names
unique_proteins = df['base_name'].unique()

# If you want as a list
unique_proteins_list = unique_proteins.tolist()

print(len(unique_proteins_list), "unique proteins found.")
print(unique_proteins_list[:10])  # Show first 10

33 unique proteins found.
['Sterol carrier protein 2', 'Hormone-sensitive lipase', 'Hedgehog protein', 'Broad substrate specificity ATP-binding cassette transporter ABCG2', 'Organic anion transporter 3', 'Lanosterol 14-alpha demethylase', 'Glucosylceramidase', 'Tyrosine-protein kinase', 'Cytochrome P450 1A', 'Acyl-coenzyme A diphosphatase NUDT19']
